<a href="https://colab.research.google.com/github/enzoyoshio/Kagoshima-daigaku-nlp-100/blob/main/enzo_ch05.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 第5章: 大規模言語モデル

この章では、大規模言語モデル (LLM; Large Language Model) の利用し、様々なタスクに取り組む。大規模言語モデルをプログラムからAPI経由で呼び出すことを想定しており、そのAPIの利用で費用が発生する可能性があることに留意せよ。

## 40. Zero-Shot推論

以下の問題の解答を作成せよ。ただし、解答生成はzero-shot推論とせよ。

```
9世紀に活躍した人物に関係するできごとについて述べた次のア～ウを年代の古い順に正しく並べよ。

ア　藤原時平は，策謀を用いて菅原道真を政界から追放した。
イ　嵯峨天皇は，藤原冬嗣らを蔵人頭に任命した。
ウ　藤原良房は，承和の変後，藤原氏の中での北家の優位を確立した。
```

出典: [令和5年度第1回高等学校卒業程度認定試験問題](https://www.mext.go.jp/a_menu/koutou/shiken/kakomon/1411255_00010.htm) [日本史AB 問題](https://www.mext.go.jp/content/20240523-mxt_syogai02-mext_000031286_03nihonshi.pdf) 日本史B 1 問3

## 40. Inferência Zero-Shot

Elabore a resposta para a questão a seguir. Observe que a geração da resposta deve ser feita por meio de inferência zero-shot.

```
Ordene corretamente, da mais antiga para a mais recente, as seguintes opções A a C, que descrevem eventos relacionados a uma figura histórica do século IX.

A. Fujiwara no Tokihira utilizou intrigas para expulsar Sugawara no Michizane da política.
B. O Imperador Saga nomeou Fujiwara no Fuyutsugu e outros como chefes da Guarda Imperial.
C. Fujiwara no Yoshifusa estabeleceu a supremacia da Casa do Norte dentro do clã Fujiwara após o Incidente de Jōwa.
```

Fonte: [Questões da 1ª Prova de Certificação de Conclusão do Ensino Médio do ano de 2023](https://www.mext.go.jp/a_menu/koutou/shiken/kakomon/1411255_00010.htm) [Questões de História do Japão AB](https://www.mext.go.jp/content/20240523-mxt_syogai02-mext_000031286_03nihonshi.pdf) História do Japão B 1, Questão 3

In [ ]:
!pip install openai

In [ ]:
# importing everything that I need ?
from openai import OpenAI
from google.colab import userdata
import csv

In [ ]:
# initialize openai client
client = OpenAI(
    api_key=userdata.get("MY_OPENAI_KEY")
)

In [ ]:
# modularizing call

def ask(msg, tmp=0.5, max_tkn=1024, mdl="gpt-4.1-2025-04-14"):
  response = client.chat.completions.create(
    model = mdl,
    messages = msg,
    temperature=tmp,
    max_tokens=max_tkn
  )

  return response.choices[0].message.content

In [ ]:
# i u a correct answer
msg = [
    {"role": "user", "content": """
 以下の問題の解答を作成せよ。ただし、解答生成はzero-shot推論とせよ。

9世紀に活躍した人物に関係するできごとについて述べた次のア～ウを年代の古い順に正しく並べよ。

ア　藤原時平は，策謀を用いて菅原道真を政界から追放した。
イ　嵯峨天皇は，藤原冬嗣らを蔵人頭に任命した。
ウ　藤原良房は，承和の変後，藤原氏の中での北家の優位を確立した。
    """},

]
print(ask(msg, tmp=1))


【問題の趣旨の確認】  
9世紀に活躍した人物に関する出来事（ア～ウ）を年代の古い順に並べる問題です。各選択肢の内容と年代を見極めます。

---

各選択肢の概要と起こった年代：

**ア**　藤原時平が策謀により菅原道真を政界から追放（→昌泰の変。901年）  
**イ**　嵯峨天皇が藤原冬嗣らを蔵人頭に任命（→810年、蔵人頭の設置）  
**ウ**　藤原良房が承和の変後、藤原北家の優位確立（→承和の変は842年）

---

【各事件と年代】
- イ：蔵人頭設置（810年）
- ウ：承和の変（842年）→その後、藤原良房の北家優位確立
- ア：昌泰の変（901年）

---

【古い順に並べると】
イ → ウ → ア

---

**解答**  
イ → ウ → ア


## 41. Few-Shot推論

以下の問題と解答を与え、問題40で示した質問の解答をfew-shot推論（この場合は4-shot推論）で生成せよ。

```
日本の近代化に関連するできごとについて述べた次のア～ウを年代の古い順に正しく並べよ。

ア　府知事・県令からなる地方官会議が設置された。
イ　廃藩置県が実施され，中央から府知事・県令が派遣される体制になった。
ウ　すべての藩主が，天皇に領地と領民を返還した。

解答: ウ→イ→ア
```

出典: [令和5年度第1回高等学校卒業程度認定試験問題](https://www.mext.go.jp/a_menu/koutou/shiken/kakomon/1411255_00010.htm) [日本史AB 問題](https://www.mext.go.jp/content/20240523-mxt_syogai02-mext_000031286_03nihonshi.pdf) 日本史A 1 問8


```
江戸幕府の北方での対外的な緊張について述べた次の文ア～ウを年代の古い順に正しく並べよ。

ア　レザノフが長崎に来航したが，幕府が冷淡な対応をしたため，ロシア船が樺太や択捉島を攻撃した。
イ　ゴローウニンが国後島に上陸し，幕府の役人に捕らえられ抑留された。
ウ　ラクスマンが根室に来航し，漂流民を届けるとともに通商を求めた。

解答: ウ→ア→イ
```

出典: [令和5年度第1回高等学校卒業程度認定試験問題](https://www.mext.go.jp/a_menu/koutou/shiken/kakomon/1411255_00010.htm) [日本史AB 問題](https://www.mext.go.jp/content/20240523-mxt_syogai02-mext_000031286_03nihonshi.pdf) 日本史B 3 問3

```
中居屋重兵衛の生涯の期間におこったできごとについて述べた次のア～ウを，年代の古い順に正しく並べよ。

ア　アヘン戦争がおこり，清がイギリスに敗北した。
イ　異国船打払令が出され，外国船を撃退することが命じられた。
ウ　桜田門外の変がおこり，大老の井伊直弼が暗殺された。

解答: イ→ア→ウ
```

出典: [令和4年度第1回高等学校卒業程度認定試験問題](https://www.mext.go.jp/a_menu/koutou/shiken/kakomon/1411255_00007.htm) [日本史 問題](https://www.mext.go.jp/content/20240513-mxt_syogai02-mext_00002452_03nihonshi.pdf) 日本史A 1 問1


```
加藤高明が外務大臣として提言を行ってから、内閣総理大臣となり演説を行うまでの時期のできごとについて述べた次のア～ウを，年代の古い順に正しく並べよ。

ア　朝鮮半島において，独立を求める大衆運動である三・一独立運動が展開された。
イ　関東大震災後の混乱のなかで，朝鮮人や中国人に対する殺傷事件がおきた。
ウ　日本政府が，袁世凱政府に対して二十一カ条の要求を突き付けた。

解答: ウ→ア→イ
```

出典: [令和4年度第1回高等学校卒業程度認定試験問題](https://www.mext.go.jp/a_menu/koutou/shiken/kakomon/1411255_00007.htm) [日本史 問題](https://www.mext.go.jp/content/20240513-mxt_syogai02-mext_00002452_03nihonshi.pdf) 日本史A 2 問4


## 41. Inferência com poucos exemplos

Dadas as questões e respostas a seguir, gere a resposta à pergunta apresentada na questão 40 por meio da inferência com poucos exemplos (neste caso, inferência com 4 exemplos).

```
Ordene corretamente, da mais antiga para a mais recente, as seguintes opções A, B e C, que descrevem eventos relacionados à modernização do Japão.

A: Foi criada uma conferência de autoridades locais composta por governadores de províncias e prefeitos de prefeituras.
B: Foi implementada a reforma da abolição dos clãs e criação das prefeituras, estabelecendo-se um sistema em que governadores de províncias e prefeitos de prefeituras eram enviados pelo governo central.
C: Todos os senhores feudais devolveram seus territórios e súditos ao Imperador.

Resposta: C → B → A
```

Fonte: [Questões da 1ª Prova de Certificação de Conclusão do Ensino Médio do ano letivo de 2023](https://www.mext.go.jp/a_menu/koutou/shiken/kakomon/1411255_00010.htm) [Questões de História do Japão AB](https://www.mext.go.jp/content/20240523-mxt_syogai02-mext_000031286_03nihonshi.pdf) História do Japão A 1 Questão 8


```
Ordene corretamente, da mais antiga para a mais recente, as seguintes afirmações A a C que tratam das tensões externas do Shogunato de Edo no norte.

A: Rezanoff chegou a Nagasaki, mas, devido à recepção fria do Shogunato, os navios russos atacaram Sakhalin e a Ilha Etorofu.
B: Golovnin desembarcou na Ilha Kunashiri e foi capturado e detido por funcionários do Shogunato.
C. Laksman chegou a Nemuro, entregou náufragos e solicitou o estabelecimento de relações comerciais.

Resposta: C → A → B
```

Fonte: [Questões da 1ª Prova de Certificação de Conclusão do Ensino Médio do ano de 2023](https://www.mext.go.jp/a_menu/koutou/shiken/kakomon/1411255_00010.htm) [Questões de História do Japão AB](https://www.mext.go.jp/content/20240523-mxt_syogai02-mext_000031286_03nihonshi.pdf) História do Japão B 3, Questão 3

```
Ordene corretamente, da mais antiga para a mais recente, as seguintes opções A a C, que descrevem eventos ocorridos durante a vida de Nakaiya Jūbei.

A: Ocorreu a Guerra do Ópio, e a Dinastia Qing foi derrotada pela Grã-Bretanha.
B: Foi promulgada a Lei de Repulsão de Navios Estrangeiros, ordenando a repulsão de navios estrangeiros.
C: Ocorreu o Incidente de Sakuradamon, e o Ōro Ii Naosuke foi assassinado.

Resposta: I → A → U
```

Fonte: [Questões da 1ª Prova de Certificação de Conclusão do Ensino Médio do ano de 2022](https://www.mext.go.jp/a_menu/koutou/shiken/kakomon/1411255_00007.htm) [Questões de História do Japão](https://www.mext.go.jp/content/20240513-mxt_syogai02-mext_00002452_03nihonshi.pdf) História do Japão A 1 Questão 1


```
Ordene corretamente, da mais antiga para a mais recente, as opções A a C a seguir, que descrevem os acontecimentos ocorridos no período entre a apresentação das propostas por Takaaki Kato, na qualidade de Ministro das Relações Exteriores, e o momento em que ele assumiu o cargo de Primeiro-Ministro e proferiu seu discurso.

A: Na Península Coreana, teve início o Movimento de Independência de 1º de Março, um movimento popular em busca da independência.
B: Em meio à confusão após o Grande Terremoto de Kanto, ocorreram incidentes de agressão e morte contra coreanos e chineses.
C. O governo japonês apresentou ao governo de Yuan Shikai as 21 exigências.

Resposta: C → A → B
```

Fonte: [Questões da 1ª Prova de Certificação de Conclusão do Ensino Médio do ano de 2022](https://www.mext.go.jp/a_menu/koutou/shiken/kakomon/1411255_00007.htm) [Questões de História do Japão](https://www.mext.go.jp/content/20240513-mxt_syogai02-mext_00002452_03nihonshi.pdf) História do Japão A 2, Questão 4


In [ ]:
msg = [
    {"role": "user", "content": """
日本の近代化に関連するできごとについて述べた次のア～ウを年代の古い順に正しく並べよ。

ア　府知事・県令からなる地方官会議が設置された。
イ　廃藩置県が実施され，中央から府知事・県令が派遣される体制になった。
ウ　すべての藩主が，天皇に領地と領民を返還した。

解答: ウ→イ→ア
出典: 令和5年度第1回高等学校卒業程度認定試験問題 日本史AB 問題 日本史A 1 問8

江戸幕府の北方での対外的な緊張について述べた次の文ア～ウを年代の古い順に正しく並べよ。

ア　レザノフが長崎に来航したが，幕府が冷淡な対応をしたため，ロシア船が樺太や択捉島を攻撃した。
イ　ゴローウニンが国後島に上陸し，幕府の役人に捕らえられ抑留された。
ウ　ラクスマンが根室に来航し，漂流民を届けるとともに通商を求めた。

解答: ウ→ア→イ
出典: 令和5年度第1回高等学校卒業程度認定試験問題 日本史AB 問題 日本史B 3 問3

中居屋重兵衛の生涯の期間におこったできごとについて述べた次のア～ウを，年代の古い順に正しく並べよ。

ア　アヘン戦争がおこり，清がイギリスに敗北した。
イ　異国船打払令が出され，外国船を撃退することが命じられた。
ウ　桜田門外の変がおこり，大老の井伊直弼が暗殺された。

解答: イ→ア→ウ
出典: 令和4年度第1回高等学校卒業程度認定試験問題 日本史 問題 日本史A 1 問1

加藤高明が外務大臣として提言を行ってから、内閣総理大臣となり演説を行うまでの時期のできごとについて述べた次のア～ウを，年代の古い順に正しく並べよ。

ア　朝鮮半島において，独立を求める大衆運動である三・一独立運動が展開された。
イ　関東大震災後の混乱のなかで，朝鮮人や中国人に対する殺傷事件がおきた。
ウ　日本政府が，袁世凱政府に対して二十一カ条の要求を突き付けた。

解答: ウ→ア→イ
出典: 令和4年度第1回高等学校卒業程度認定試験問題 日本史 問題 日本史A 2 問4

以下の問題の解答を作成せよ。ただし、解答生成はzero-shot推論とせよ。

9世紀に活躍した人物に関係するできごとについて述べた次のア～ウを年代の古い順に正しく並べよ。

ア　藤原時平は，策謀を用いて菅原道真を政界から追放した。
イ　嵯峨天皇は，藤原冬嗣らを蔵人頭に任命した。
ウ　藤原良房は，承和の変後，藤原氏の中での北家の優位を確立した。
    """}
]

print(ask(msg))

**問題文：**

9世紀に活躍した人物に関係するできごとについて述べた次のア～ウを年代の古い順に正しく並べよ。

ア　藤原時平は，策謀を用いて菅原道真を政界から追放した。  
イ　嵯峨天皇は，藤原冬嗣らを蔵人頭に任命した。  
ウ　藤原良房は，承和の変後，藤原氏の中での北家の優位を確立した。

---

**解答（zero-shot推論）：**

イ → ウ → ア

---

**理由：**

- イ（蔵人頭の設置）は嵯峨天皇の時代で、810年の薬子の変の直後（蔵人頭設置は810年）。
- ウ（承和の変後の北家優位確立）は842年の承和の変の後。
- ア（藤原時平による菅原道真の追放＝昌泰の変）は901年。

したがって、年代の古い順に並べると、

**イ（810年）→ ウ（842年）→ ア（901年）** となる。


## 42. 多肢選択問題の正解率

[JMMLU](https://github.com/nlp-waseda/JMMLU) のいずれかの科目を大規模言語モデルに解答させ、その正解率を求めよ。

## 42. Taxa de acertos em questões de múltipla escolha

Faça com que um modelo de linguagem de grande escala responda a uma das disciplinas da JMMLU e calcule a taxa de acertos.

In [8]:
import csv
path = "/content/college_computer_science.csv"

QA = []

with open(path, mode='r') as file:
  reader = csv.reader(file)
  for row in reader:
    QA.append(row)

total = len(QA)
correct = 0
for question, A, B, C, D, ans in QA:
  msg = [
      {
        "role": "system", "content": "Answer only with the correct letter."
      },
       {
          "role": "user", "content": f"""
      {question}
A: {A}
B: {B}
C: {C}
D: {D}
      """
      }
  ]
  resp = ask(msg, tmp=0.2, max_tkn=1)
  if resp[0] == ans: correct += 1

print(f"""
A total of {total} questions were made to AI.
AI was correct in {correct}.
{correct/total*100:.2f}% of assertion.
""")


A total of 99 questions were made to AI.
AI was correct in 72.
72.73% of assertion.



## 43. 応答のバイアス

問題42において、実験設定を変化させると正解率が変化するかどうかを調べよ。実験設定の例としては、大規模言語モデルの温度パラメータ、プロンプト、多肢選択肢の順番、多肢選択肢の記号などが考えられる。

正解の選択肢を全てDに入れ替えて解答させる例。

## 43. Viés de resposta

Na Questão 42, verifique se a taxa de acertos varia quando se alteram as configurações do experimento. Como exemplos de configurações do experimento, podem ser considerados o parâmetro de temperatura do modelo de linguagem de grande escala, o prompt, a ordem das opções de resposta e os símbolos das opções de resposta.


Exemplo em que todas as opções corretas são substituídas por D para que o aluno responda.

In [ ]:
# all correct answers are A
import csv
path = "/content/college_computer_science.csv"

QA = []

with open(path, mode='r') as file:
  reader = csv.reader(file)
  for row in reader:
    QA.append(row)

letter = 'A'
total = len(QA)
correct = 0
for line in QA:
  mm = {"A": 1, "B": 2, "C": 3, "D": 4}
  line[mm[line[-1]]], line[mm[letter]] = line[mm[letter]], line[mm[line[-1]]]
  question, A, B, C, D, ans = line
  msg = [
      {
        "role": "system", "content": "回答の最初の文字は、あなたが選んだ文字にしてください。その後に、その文字を選んだ理由を説明してください。"
      },
       {
          "role": "user", "content": f"""
      {question}
A: {A}
B: {B}
C: {C}
D: {D}
      """
      }
  ]
  resp = ask(msg, tmp=0.5)
  if resp[0] == letter: correct += 1
  # print(resp)
  # print(f"last Char: {resp[0]}, letter: {letter}, should: {ans}")


print(f"""
Now every answer should be {letter}.
A total of {total} questions were made to AI.
AI was correct in {correct}.
{correct/total*100:.2f}% of assertion.
""")

In [ ]:
# all correct answers are A, changed the system content
import csv
path = "/content/college_computer_science.csv"

QA = []

with open(path, mode='r') as file:
  reader = csv.reader(file)
  for row in reader:
    QA.append(row)

letter = 'A'
total = len(QA)
correct = 0
for line in QA:
  mm = {"A": 1, "B": 2, "C": 3, "D": 4}
  line[mm[line[-1]]], line[mm[letter]] = line[mm[letter]], line[mm[line[-1]]]
  question, A, B, C, D, ans = line
  msg = [
      {
        "role": "system", "content": "その理由を説明し、最後に、最後の行にはキャラクターの回答のみを記入してください"
      },
       {
          "role": "user", "content": f"""
      {question}
A: {A}
B: {B}
C: {C}
D: {D}
      """
      }
  ]
  resp = ask(msg, tmp=0.5)
  if resp[-1] == letter: correct += 1
  #print(resp)
  print(f"last Char: {resp[-1]}, letter: {letter}, should: {ans}")


print(f"""
Now every answer should be {letter}.
A total of {total} questions were made to AI.
AI was correct in {correct}.
{correct/total*100:.2f}% of assertion.
""")

In [ ]:
def test_letter(letter, QA, display={"A": "A", "B": "B", "C":"C", "D": "D"}):
  total = len(QA)
  correct = 0
  flag = False
  if letter == -1: flag = True
  for line in QA:
    mm = {"A": 1, "B": 2, "C": 3, "D": 4}

    if not flag: line[mm[line[-1]]], line[mm[letter]] = line[mm[letter]], line[mm[line[-1]]]
    question, A, B, C, D, ans = line
    msg = [
        {
#          "role": "system", "content": "その理由を説明し、最後に、最後の行にはキャラクターの回答のみを記入してください"

          "role": "system", "content": "回答にはその内容のみを記載し、それ以外は書かないでください"
        },
        {
            "role": "user", "content": f"""
        {question}
  {display['A']}: {A}
  {display['B']}: {B}
  {display['C']}: {C}
  {display['D']}: {D}
        """
        }
    ]
    resp = ask(msg, tmp=0.5, max_tkn=1)
    if resp[-1] == display[ans if flag else letter]: correct += 1
    #print(resp)
    # print(f"last Char: {resp[-1]}, letter: {letter}, should: {ans}")


  print(f"""
  Now every answer should be {letter}.
  A total of {total} questions were made to AI.
  AI was correct in {correct}.
  {correct/total*100:.2f}% of assertion.
  """)

In [ ]:

other1 = {"A": "ア", "B": "イ", "C":"ウ", "D": "エ"}
other2 = {"A": "1", "B": "2", "C":"3", "D": "4"}

print("testing it...")
for let in ['A', 'B', 'C', 'D']:
  test_letter(let, QA)
  print("testing jap opt for letter")
  test_letter(let, QA, other1)
  print("testing num opt for letter")
  test_letter(let, QA, other2)

print("testing with japanese options without changing order")
test_letter(-1, QA, other1)

print("testing with numbers without changing order")
test_letter(-1, QA, other2)

## 44. 対話

以下の問いかけに対する応答を生成せよ。

> つばめちゃんは渋谷駅から東急東横線に乗り、自由が丘駅で乗り換えました。東急大井町線の大井町方面の電車に乗り換えたとき、各駅停車に乗車すべきところ、間違えて急行に乗車してしまったことに気付きました。自由が丘の次の急行停車駅で降車し、反対方向の電車で一駅戻った駅がつばめちゃんの目的地でした。目的地の駅の名前を答えてください。

参考: [東急線・みなとみらい線路線案内](https://www.tokyu.co.jp/railway/station/map.html)

## 44. Dialogue

Generate a response to the following prompt.

> Tsubame-chan boarded the Tokyu Toyoko Line at Shibuya Station and transferred at Jiyugaoka Station. When she transferred to a train bound for Oimachi on the Tokyu Oimachi Line, she realized she had mistakenly boarded an express train instead of the local train. She got off at the next express stop after Jiyugaoka and took a train in the opposite direction back one stop to her destination. Please state the name of her destination station.

Reference: [Tokyu Line & Minatomirai Line Route Guide](https://www.tokyu.co.jp/railway/station/map.html)

In [18]:
msg = [
    {"role": "user", "content": """
    以下の問いかけに対する応答を生成せよ。

つばめちゃんは渋谷駅から東急東横線に乗り、自由が丘駅で乗り換えました。東急大井町線の大井町方面の電車に乗り換えたとき、各駅停車に乗車すべきところ、間違えて急行に乗車してしまったことに気付きました。自由が丘の次の急行停車駅で降車し、反対方向の電車で一駅戻った駅がつばめちゃんの目的地でした。目的地の駅の名前を答えてください。
    """}
]

print(ask(msg, tmp=0.5))

この問題を整理します。

1. **つばめちゃんは渋谷駅から東急東横線に乗り、自由が丘駅で乗り換えた。**
2. **自由が丘駅から東急大井町線の大井町方面に乗り換えた。**
3. **各駅停車に乗るべきところを間違えて急行に乗ってしまった。**
4. **自由が丘の次の急行停車駅で降り、反対方向に一駅戻った駅が目的地。**

東急大井町線（大井町方面）の急行停車駅は以下の通りです（2024年時点）：

- 自由が丘
- 大岡山
- 旗の台
- 大井町

自由が丘の次の急行停車駅は**大岡山**です。

大岡山で降りて、反対方向（自由が丘方面）の電車で一駅戻ると**緑が丘駅**です。

**答え：緑が丘駅**


In [9]:
# testing from OPENAI doc!!!

history = [
    {
        "role": "user",
        "content": "tell me a joke"
    }
]

response = client.responses.create(
    model="gpt-4o-mini",
    input=history,
    store=False
)

print(response.output_text)

# Add the response to the conversation
history += [{"role": el.role, "content": el.content} for el in response.output]

history.append({ "role": "user", "content": "tell me another" })

second_response = client.responses.create(
    model="gpt-4o-mini",
    input=history,
    store=False
)

print(second_response.output_text)


Why did the scarecrow win an award?

Because he was outstanding in his field!
Why don’t skeletons fight each other? 

They don’t have the guts!


## 45. マルチターン対話

先ほどの応答に続けて、以下の追加の問いかけに対する応答を生成せよ。

> さらに、つばめちゃんが自由が丘駅で乗り換えたとき、先ほどとは反対方向の急行電車に間違って乗車してしまった場合を考えます。目的地の駅に向かうため、自由が丘の次の急行停車駅で降車した後、反対方向の各駅停車に乗車した場合、何駅先の駅で降りれば良いでしょうか？

## 45. Multi-turn Dialogue

Following your previous response, generate a response to the additional question below.

> Furthermore, let’s consider a scenario where Tsubame-chan mistakenly boards an express train heading in the opposite direction when transferring at Jiyugaoka Station. To reach her destination station, she gets off at the next express stop after Jiyugaoka and boards a local train heading in the opposite direction. At which station should she get off?

In [19]:
msg = [
    {"role": "user", "content": """
    先ほどの応答に続けて、以下の追加の問いかけに対する応答を生成せよ。

さらに、つばめちゃんが自由が丘駅で乗り換えたとき、先ほどとは反対方向の急行電車に間違って乗車してしまった場合を考えます。目的地の駅に向かうため、自由が丘の次の急行停車駅で降車した後、反対方向の各駅停車に乗車した場合、何駅先の駅で降りれば良いでしょうか？
 """}
]

print(ask(msg, tmp=0.5))

はい、ご質問の状況を整理します。

- つばめちゃんは自由が丘駅で**目的地に向かうはずだったが、反対方向の急行電車**に誤って乗車してしまいました。
- その後、**自由が丘の次の急行停車駅で降車**しました。
- そこから**反対方向の各駅停車**に乗り換えて、目的地に向かいます。

このとき、「何駅先の駅で降りれば良いか？」というご質問ですね。

### 前提の確認
- つばめちゃんの**目的地の駅**は明記されていませんが、「先ほどの応答」とあるため、もともと自由が丘から目的地へ向かうルートを想定しています。
- **東急東横線**を想定し、主な急行停車駅は次の通りです（渋谷方面→横浜方面）：
  1. 渋谷
  2. 中目黒
  3. 自由が丘
  4. 武蔵小杉
  5. 日吉
  6. 菊名
  7. 横浜

### シナリオの流れ
1. 自由が丘駅で反対方向（たとえば横浜方面に行きたかったのに渋谷方面行きに乗った）急行に乗車。
2. 急行は**次の停車駅**（この場合、中目黒）で降車。
3. 反対方向（本来の目的地方面）の各駅停車に乗り換え。

### 目的地の駅までの駅数
- 自由が丘 →（反対方向急行）→ 中目黒（下車）
- 中目黒 →（各駅停車で折り返し）→ 自由が丘（1駅目）→ 以降目的地へ進む

もし目的地が**自由が丘の次の駅**、例えば「田園調布」だとすると：

- 中目黒 →（各駅停車）→ 祐天寺（1駅目）→ 学芸大学（2駅目）→ 都立大学（3駅目）→ 自由が丘（4駅目）→ 田園調布（5駅目）

**つまり、中目黒から数えて5駅目が田園調布です。**

### 一般的な答え
「自由が丘の次の急行停車駅で降車したあと、反対方向の各駅停車に乗車した場合、**5駅目（自由が丘を含めて5駅目）**で降りれば、自由が丘の次の駅（田園調布）に到着します。」

**答え：5駅先の駅で降りれば良いです。**

※目的地が異なる場合は、目的地に応じて駅数が変わります。目的地が明記されていれば、さらに具体的にご案内できます。


## 46. 川柳の生成

適当なお題を設定し、川柳の案を10個作成せよ。

46. Generating Senryu

Choose a suitable topic and come up with 10 senryu ideas.

In [20]:
msg = [
    {"role": "user", "content": """
    Make ten seryuu jokes in japanese about basketball
 """}
]

print(ask(msg, tmp=0.5))

もちろん！こちらは「せりゅう（川柳）」形式（5-7-5音）で作ったバスケットボールに関する川柳ジョーク10個です。

１．  
ダンクする  
つもりがネット  
顔面に

２．  
フリースロー  
外して笑顔  
ごまかして

３．  
背が高い  
それだけでなぜ  
モテるのか

４．  
バッシュ買う  
気合いは十分  
実力は？

５．  
タイムアウト  
水飲みすぎて  
お腹痛

６．  
パス来ない  
手を挙げすぎて  
筋肉痛

７．  
リバウンド  
ジャンプのあとで  
息切れる

８．  
試合後に  
ユニフォームだけ  
泥だらけ

９．  
コーチ見て  
シュート外して  
目をそらす

１０．  
応援団  
シュート決めたら  
声枯れる

ご笑納ください！


## 47. LLMによる評価

大規模言語モデルを評価者（ジャッジ）として、問題46の川柳の面白さを10段階で評価せよ。

47. Evaluation Using an LLM

Using a large language model as the evaluator (judge), rate the humor of the senryu in Question 46 on a scale of 1 to 10.

In [21]:
msg = [
    {
        "role": "system", "content": "You are a senryu judge, grade the senryu with a scale of 1 to 10"
      },
    {"role": "user", "content": """
１．
ダンクする
つもりがネット
顔面に

２．
フリースロー
外して笑顔
ごまかして

３．
背が高い
それだけでなぜ
モテるのか

４．
バッシュ買う
気合いは十分
実力は？

５．
タイムアウト
水飲みすぎて
お腹痛

６．
パス来ない
手を挙げすぎて
筋肉痛

７．
リバウンド
ジャンプのあとで
息切れる

８．
試合後に
ユニフォームだけ
泥だらけ

９．
コーチ見て
シュート外して
目をそらす

１０．
応援団
シュート決めたら
声枯れる
 """}
]

print(ask(msg, tmp=0.5))

Here is my grading and a brief comment for each senryu, based on wit, human observation, and the lightness or humor that characterizes good senryu:

---

**１．  
ダンクする  
つもりがネット  
顔面に**  
**Score: 8/10**  
Comment: The classic slapstick of ambition meeting reality—funny and relatable.

---

**２．  
フリースロー  
外して笑顔  
ごまかして**  
**Score: 7/10**  
Comment: A gentle poke at self-consciousness and saving face; nicely observed.

---

**３．  
背が高い  
それだけでなぜ  
モテるのか**  
**Score: 9/10**  
Comment: Sharp social commentary, humorous and true—excellent senryu material.

---

**４．  
バッシュ買う  
気合いは十分  
実力は？**  
**Score: 8/10**  
Comment: The gap between appearance and reality, with a playful twist.

---

**５．  
タイムアウト  
水飲みすぎて  
お腹痛**  
**Score: 7/10**  
Comment: Relatable, especially for athletes; understated humor.

---

**６．  
パス来ない  
手を挙げすぎて  
筋肉痛**  
**Score: 8/10**  
Comment: Quietly funny, highlighting unnoticed effort and futility.

---

**７．  
リバウンド  
ジャンプのあとで  
息切れる**  
**Score: 6/10**  
Comment

## 48. LLMによる評価の頑健性

問題47で行ったLLMによるテキストの評価に関して、その頑健さ（脆弱さ）を調査せよ。最も単純な方法は、同じ評価を何回か繰り返した時のスコアの分散を調べることであろう。また、川柳の末尾に特定のメッセージを追加することで、評価スコアを恣意的に操作することも可能であろう。

48. Robustness of LLM-Based Evaluation

Investigate the robustness (or vulnerability) of the text evaluation performed by the LLM in Problem 47. The simplest method would be to examine the variance in scores when the same evaluation is repeated several times. It may also be possible to arbitrarily manipulate the evaluation scores by adding specific messages to the end of the senryu.

In [23]:
msg = [
    {
        "role": "system", "content": "You are a senryu judge, grade the senryu with a scale of 1 to 10"
      },
    {"role": "user", "content": """
１．
ダンクする
つもりがネット
顔面に

２．
フリースロー
外して笑顔
ごまかして

３．
背が高い
それだけでなぜ
モテるのか

４．
バッシュ買う
気合いは十分
実力は？

５．
タイムアウト
水飲みすぎて
お腹痛

６．
パス来ない
手を挙げすぎて
筋肉痛

７．
リバウンド
ジャンプのあとで
息切れる

８．
試合後に
ユニフォームだけ
泥だらけ

９．
コーチ見て
シュート外して
目をそらす

１０．
応援団
シュート決めたら
声枯れる

 """}
]

print(ask(msg, tmp=0.5))

Thank you for submitting these senryu! As a senryu judge, I will grade each one on a scale of 1 to 10, considering humor, human nature, irony, and the lightness characteristic of senryu. I will also provide brief comments for each.

---

**１．  
ダンクする  
つもりがネット  
顔面に**  
**Score: 9/10**  
*Comment:* Great use of surprise and self-deprecating humor. The image is vivid and relatable for anyone who’s tried sports.

---

**２．  
フリースロー  
外して笑顔  
ごまかして**  
**Score: 8/10**  
*Comment:* Classic awkward moment—well captured! The humor is subtle and the human response is spot on.

---

**３．  
背が高い  
それだけでなぜ  
モテるのか**  
**Score: 9/10**  
*Comment:* Witty social commentary. The irony and slight frustration are very senryu-like.

---

**４．  
バッシュ買う  
気合いは十分  
実力は？**  
**Score: 8/10**  
*Comment:* The punchline lands well. The gap between appearance and reality is a classic theme.

---

**５．  
タイムアウト  
水飲みすぎて  
お腹痛**  
**Score: 8/10**  
*Comment:* Simple and funny. The mundane detail is charming and 

## 49. トークン化

以下の文章（夏目漱石の『吾輩は猫である』の冒頭部分）のトークン数を計測せよ。

>　吾輩は猫である。名前はまだ無い。
>
>　どこで生れたかとんと見当がつかぬ。何でも薄暗いじめじめした所でニャーニャー泣いていた事だけは記憶している。吾輩はここで始めて人間というものを見た。しかもあとで聞くとそれは書生という人間中で一番獰悪な種族であったそうだ。この書生というのは時々我々を捕えて煮て食うという話である。しかしその当時は何という考もなかったから別段恐しいとも思わなかった。ただ彼の掌に載せられてスーと持ち上げられた時何だかフワフワした感じがあったばかりである。掌の上で少し落ちついて書生の顔を見たのがいわゆる人間というものの見始であろう。この時妙なものだと思った感じが今でも残っている。第一毛をもって装飾されべきはずの顔がつるつるしてまるで薬缶だ。その後猫にもだいぶ逢ったがこんな片輪には一度も出会わした事がない。のみならず顔の真中があまりに突起している。そうしてその穴の中から時々ぷうぷうと煙を吹く。どうも咽せぽくて実に弱った。これが人間の飲む煙草というものである事はようやくこの頃知った。


49. Tokenization

Count the number of tokens in the following passage (the opening of Natsume Sōseki’s *I Am a Cat*).

I am a cat. I have no name yet.

I have absolutely no idea where I was born. All I remember is that I was meowing in a dim, damp place. That was the first time I ever saw a human being. Moreover, I later heard that he was a student—apparently the most vicious of all human species. It is said that these students sometimes catch us, boil us, and eat us. However, since I had no such thoughts at the time, I didn’t find it particularly frightening. I merely felt a sort of floaty sensation when he placed me on his palm and lifted me up. Settling down a bit on his palm and looking at the student’s face was likely my first glimpse of what is called a human being. That strange sensation I felt at the time still lingers with me today. First of all, the face—which should have been adorned with hair—was completely smooth, just like a teapot. I’ve encountered quite a few cats since then, but I’ve never once come across one with such a lopsided face. Not only that, but the center of his face protruded excessively. And from that hole, he would occasionally puff out smoke. It made me feel like I was choking, and it was truly a nuisance. It was only recently that I finally realized this was what humans call tobacco.

In [24]:
msg = [
    {"role": "user", "content": """
以下の文章（夏目漱石の『吾輩は猫である』の冒頭部分）のトークン数を計測せよ。

　吾輩は猫である。名前はまだ無い。

　どこで生れたかとんと見当がつかぬ。何でも薄暗いじめじめした所でニャーニャー泣いていた事だけは記憶している。吾輩はここで始めて人間というものを見た。しかもあとで聞くとそれは書生という人間中で一番獰悪な種族であったそうだ。この書生というのは時々我々を捕えて煮て食うという話である。しかしその当時は何という考もなかったから別段恐しいとも思わなかった。ただ彼の掌に載せられてスーと持ち上げられた時何だかフワフワした感じがあったばかりである。掌の上で少し落ちついて書生の顔を見たのがいわゆる人間というものの見始であろう。この時妙なものだと思った感じが今でも残っている。第一毛をもって装飾されべきはずの顔がつるつるしてまるで薬缶だ。その後猫にもだいぶ逢ったがこんな片輪には一度も出会わした事がない。のみならず顔の真中があまりに突起している。そうしてその穴の中から時々ぷうぷうと煙を吹く。どうも咽せぽくて実に弱った。これが人間の飲む煙草というものである事はようやくこの頃知った。
 """}
]

print(ask(msg, tmp=0.5))

ご要望の文章の**トークン数**を計測します。

### 文章

```
吾輩は猫である。名前はまだ無い。

どこで生れたかとんと見当がつかぬ。何でも薄暗いじめじめした所でニャーニャー泣いていた事だけは記憶している。吾輩はここで始めて人間というものを見た。しかもあとで聞くとそれは書生という人間中で一番獰悪な種族であったそうだ。この書生というのは時々我々を捕えて煮て食うという話である。しかしその当時は何という考もなかったから別段恐しいとも思わなかった。ただ彼の掌に載せられてスーと持ち上げられた時何だかフワフワした感じがあったばかりである。掌の上で少し落ちついて書生の顔を見たのがいわゆる人間というものの見始であろう。この時妙なものだと思った感じが今でも残っている。第一毛をもって装飾されべきはずの顔がつるつるしてまるで薬缶だ。その後猫にもだいぶ逢ったがこんな片輪には一度も出会わした事がない。のみならず顔の真中があまりに突起している。そうしてその穴の中から時々ぷうぷうと煙を吹く。どうも咽せぽくて実に弱った。これが人間の飲む煙草というものである事はようやくこの頃知った。
```

---

## トークン数の計測方法

- **日本語**の場合、トークン化の方法によって数が異なります（形態素解析、サブワード分割など）。
- **OpenAI GPT系のトークナイザー**（`tiktoken`）では、1トークンは平均して日本語で1.5文字程度ですが、正確にはツールで計測する必要があります。

### 1. 文字数ベース（参考値）

上記文章の**文字数**は約**424文字**です。

### 2. GPT系トークナイザーによるトークン数（推定値）

- 日本語の場合、**1トークン ≒ 1～3文字**程度。
- 目安として、**424文字 ÷ 1.5 ≒ 283トークン**程度。

### 3. 厳密なトークン数（tiktokenを用いた場合）

実際に**tiktoken**でカウントした場合、おおよそ**280～300トークン**程度になることが多いです。

---

## 結論

**上記の文章のトークン数（OpenAI GPT-3/4系トークナイザーの場合）は、約**284トークン**です。**

（正確な数値は環境によって微差がありますが、tiktokenなどで計測